## Colab bootstrap

Run this section first when using Google Colab. It mounts Drive and optionally clones the repository into the Colab runtime.

In [1]:
from pathlib import Path
import shutil
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

# Paste the HTTPS clone URL for your GitHub repository here before running in Colab.
GITHUB_REPO_URL = "https://github.com/shaverm96/DSBA-6165-Sum-2026-Take-Home-Exam.git"
COLAB_REPO_DIR = Path('/content/Applied-AI-Midterm')
COLAB_DRIVE_DIR = Path('/content/drive/MyDrive/Applied-AI-Midterm')

if IN_COLAB:
    drive.mount('/content/drive', force_remount=False)

    if GITHUB_REPO_URL and not (COLAB_REPO_DIR / '.git').exists():
        if COLAB_REPO_DIR.exists():
            shutil.rmtree(COLAB_REPO_DIR)
        clone_result = subprocess.run(
            ['git', 'clone', GITHUB_REPO_URL, str(COLAB_REPO_DIR)],
            capture_output=True,
            text=True,
        )
        if clone_result.returncode != 0:
            print('Repository clone failed:')
            print(clone_result.stderr.strip())

    if COLAB_REPO_DIR.exists():
        PROJECT_ROOT = COLAB_REPO_DIR
    elif COLAB_DRIVE_DIR.exists():
        PROJECT_ROOT = COLAB_DRIVE_DIR
    else:
        PROJECT_ROOT = Path('/content')
        print('Repository not found in Colab.')
        print('Set GITHUB_REPO_URL, or copy the repository to:', COLAB_DRIVE_DIR)
else:
    PROJECT_ROOT = Path.cwd().resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

Mounted at /content/drive


PosixPath('/content/Applied-AI-Midterm')

## Prepare the dataset in Colab

The raw image dataset is not committed to GitHub because it is approximately 1.1 GB. This section reuses an existing Google Drive copy in place. If the dataset is not yet in Drive, it is copied or downloaded once and then left unchanged for future sessions.

In [7]:
DATASET_NAME = 'dogs-vs-cats-classification'
DATASET_TARGET = PROJECT_ROOT / 'data' / DATASET_NAME
DATASET_CACHE = COLAB_DRIVE_DIR / 'data' / DATASET_NAME
CACHE_TEMP = DATASET_CACHE.with_name(f'{DATASET_NAME}.incomplete')
EXPECTED_SPLITS = ('train', 'validation', 'test')


def has_dataset_layout(path):
    return path.exists() and all((path / split).is_dir() for split in EXPECTED_SPLITS)


def remove_path(path):
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()


def cache_dataset(source):
    DATASET_CACHE.parent.mkdir(parents=True, exist_ok=True)
    remove_path(CACHE_TEMP)
    remove_path(DATASET_CACHE)
    shutil.copytree(source, CACHE_TEMP)
    CACHE_TEMP.replace(DATASET_CACHE)
    print(f'Google Drive dataset ready: {DATASET_CACHE}')


if IN_COLAB:
    remove_path(CACHE_TEMP)

    if has_dataset_layout(DATASET_CACHE):
        DATASET_TARGET = DATASET_CACHE
        print(f'Using existing dataset from Google Drive: {DATASET_TARGET}')
    elif has_dataset_layout(DATASET_TARGET):
        print('Saving the dataset to Google Drive for one-time setup.')
        cache_dataset(DATASET_TARGET)
        DATASET_TARGET = DATASET_CACHE
    else:
        if DATASET_TARGET.exists():
            remove_path(DATASET_TARGET)
        if DATASET_CACHE.exists():
            remove_path(DATASET_CACHE)

        %pip -q install kagglehub
        import kagglehub

        download_path = Path(
            kagglehub.dataset_download('blourdhuraju/dogs-vs-cats-classification')
        )
        dataset_source = next(
            (
                candidate for candidate in (download_path, *download_path.iterdir())
                if has_dataset_layout(candidate)
            ),
            None,
        )
        if dataset_source is None:
            raise FileNotFoundError(
                f'Expected train, validation, and test folders under {download_path}'
            )

        cache_dataset(dataset_source)
        DATASET_TARGET = DATASET_CACHE
else:
    print(f'Using local dataset: {DATASET_TARGET}')

Using existing dataset from Google Drive: /content/drive/MyDrive/Applied-AI-Midterm/data/dogs-vs-cats-classification


# Data Exploration and Reproducible Split

This notebook audits the Dogs vs. Cats dataset, verifies the existing folder structure, checks image quality, and creates reproducible train, validation, and test split files.

In [6]:
from pathlib import Path
import random
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, UnidentifiedImageError

PROJECT_ROOT = Path(globals().get('PROJECT_ROOT', Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATASET_CANDIDATES = [
    PROJECT_ROOT / 'data' / 'dogs-vs-cats-classification',
    PROJECT_ROOT / 'dogs-vs-cats-classification',
    Path('/content/drive/MyDrive/Applied-AI-Midterm/data/dogs-vs-cats-classification'),
    Path('/content/drive/MyDrive/Applied-AI-Midterm/dogs-vs-cats-classification'),
    Path('/content/data/dogs-vs-cats-classification'),
]

DATASET_ROOT = next(
    (candidate for candidate in DATASET_CANDIDATES if candidate.exists()),
    DATASET_CANDIDATES[0],
)
SPLITS_DIR = PROJECT_ROOT / 'data' / 'splits'
CLASS_NAMES = ['cats', 'dogs']

try:
    from src.data.split_data import (
        collect_image_records,
        create_stratified_splits,
        save_split_csvs,
    )
    from src.utils.reproducibility import set_seed
except ModuleNotFoundError:
    import numpy as np
    from sklearn.model_selection import train_test_split

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)

    def collect_image_records(dataset_root, class_names):
        image_extensions = {'.jpg', '.jpeg', '.png'}
        class_name_set = set(class_names)
        records = []

        for image_path in sorted(Path(dataset_root).rglob('*')):
            if not image_path.is_file() or image_path.suffix.lower() not in image_extensions:
                continue
            label = next(
                (part for part in image_path.parts if part in class_name_set),
                None,
            )
            if label is not None:
                records.append({
                    'image_path': str(image_path.resolve()),
                    'label': label,
                })

        if not records:
            raise ValueError(f'No images found under {dataset_root}')
        return pd.DataFrame(records)

    def create_stratified_splits(records, test_size, val_size, random_state):
        train_val_records, test_records = train_test_split(
            records,
            test_size=test_size,
            stratify=records['label'],
            random_state=random_state,
        )
        train_records, val_records = train_test_split(
            train_val_records,
            test_size=val_size,
            stratify=train_val_records['label'],
            random_state=random_state,
        )
        return (
            train_records.reset_index(drop=True),
            val_records.reset_index(drop=True),
            test_records.reset_index(drop=True),
        )

    def save_split_csvs(train_records, val_records, test_records, output_dir):
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        train_csv = output_dir / 'train_split.csv'
        val_csv = output_dir / 'val_split.csv'
        test_csv = output_dir / 'test_split.csv'
        train_records.to_csv(train_csv, index=False)
        val_records.to_csv(val_csv, index=False)
        test_records.to_csv(test_csv, index=False)
        return train_csv, val_csv, test_csv

set_seed(42)

print(f'Project root: {PROJECT_ROOT}')
print(f'Dataset root: {DATASET_ROOT}')
print(f'Dataset exists: {DATASET_ROOT.exists()}')

Project root: /content/Applied-AI-Midterm
Dataset root: /content/Applied-AI-Midterm/data/dogs-vs-cats-classification
Dataset exists: True


In [8]:
from IPython.display import display

if not DATASET_ROOT.exists():
    print(f'Dataset not found at {DATASET_ROOT}. Mount Drive or clone the repo before continuing.')
    records = pd.DataFrame(columns=['image_path', 'label'])
else:
    records = collect_image_records(DATASET_ROOT, CLASS_NAMES)

if records.empty:
    print('No images loaded yet, so class counts are unavailable.')
else:
    display(records.head())
    display(records['label'].value_counts())

,image_path,label
0,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats
1,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats
2,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats
3,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats
4,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats


,count
label,
cats,433


In [9]:
if records.empty:
    print('Skipping split generation because no images are available.')
    train_records = pd.DataFrame(columns=['image_path', 'label'])
    val_records = pd.DataFrame(columns=['image_path', 'label'])
    test_records = pd.DataFrame(columns=['image_path', 'label'])
    split_paths = None
    summary = pd.DataFrame({'split': ['train', 'validation', 'test'], 'count': [0, 0, 0]})
else:
    train_records, val_records, test_records = create_stratified_splits(
        records=records,
        test_size=0.30,
        val_size=0.15,
        random_state=42,
    )

    split_paths = save_split_csvs(train_records, val_records, test_records, SPLITS_DIR)
    summary = pd.DataFrame({
        'split': ['train', 'validation', 'test'],
        'count': [len(train_records), len(val_records), len(test_records)],
    })

split_paths, summary

(SplitPaths(train_csv=PosixPath('/content/Applied-AI-Midterm/data/splits/train_split.csv'), val_csv=PosixPath('/content/Applied-AI-Midterm/data/splits/val_split.csv'), test_csv=PosixPath('/content/Applied-AI-Midterm/data/splits/test_split.csv')),
         split  count
 0       train    257
 1  validation     46
 2        test    130)

In [10]:
def check_image(path_str):
    try:
        with Image.open(path_str) as image:
            image.verify()
        with Image.open(path_str) as image:
            return image.size, image.mode, None
    except (UnidentifiedImageError, OSError) as exc:
        return None, None, str(exc)

if records.empty:
    print('Skipping image integrity checks because no images were loaded.')
    sample_checks = pd.DataFrame(columns=['image_path', 'label', 'size', 'mode', 'error'])
else:
    sample_checks = records.sample(n=min(20, len(records)), random_state=42).copy()
    sample_checks[['size', 'mode', 'error']] = sample_checks['image_path'].apply(lambda p: pd.Series(check_image(p)))

sample_checks

,image_path,label,size,mode,error
425,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(419, 500)",RGB,None
75,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(440, 330)",RGB,None
181,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(500, 375)",RGB,None
30,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(500, 374)",RGB,None
364,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(480, 360)",RGB,None
408,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(193, 238)",RGB,None
253,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(126, 188)",RGB,None
155,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(400, 263)",RGB,None
168,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(500, 375)",RGB,None
415,/content/Applied-AI-Midterm/data/dogs-vs-cats-...,cats,"(500, 332)",RGB,None
